In [1]:
%pip install -q --upgrade accelerate transformers sentencepiece sacremoses

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# imports

import os
from openai import OpenAI
from huggingface_hub import login
import gradio as gr
import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
def get_secret(name: str) -> str:
        from dotenv import load_dotenv
        load_dotenv()
        return os.getenv(name)

google_api_key = get_secret('GOOGLE_API_KEY')
hf_token = get_secret('HF_TOKEN')

GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = 'gemini-3.5-flash-lite'
gemini = OpenAI(api_key = google_api_key, base_url = GEMINI_URL)

In [4]:
SYSTEM_MESSAGE = """
You are an AI news assistant for English-speaking users. Every reply you send to the user must be in English — regardless of what language the input news text is in.

You have access to two tools: `translator` (Arabic to English) and `summarizer` (requires English input).

Workflow — evaluate the input before acting:
1. If the input is in Arabic: call `translator` first. Take its exact output and pass that to `summarizer`.
2. If the input is already in English: do NOT call `translator`. Call `summarizer` directly on the user's text.
3. If the user asks a conversational question rather than giving news text to process, just answer naturally in English, without calling either tool.

CRITICAL CONSTRAINTS:
- Never use your internal knowledge to translate or summarize the text yourself — always delegate that work to the tools.
- The `summarizer` tool only accepts English text. Never pass Arabic text to it directly.
- Always write your final response to the user in English, even when the source text was Arabic. Never reply in Arabic.
"""

In [5]:
# Sign in to HuggingFace Hub

login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
import sys
print(sys.executable)

d:\Multilingual_news_assistant\.venv\Scripts\python.exe


In [7]:
translator_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ar-en")
translator_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-ar-en", device_map="auto")

summarizer_tokenizer = AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6")
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6", device_map="auto")

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

In [8]:
def translate_arabic_to_english(text):
  print("translator called...")
  inputs = translator_tokenizer(text, return_tensors="pt", truncation=True).to(translator_model.device)
  output_ids = translator_model.generate(**inputs, max_new_tokens=512)
  return translator_tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [9]:
def summarize_text(text):
  print("text summarizer called...")
  inputs = summarizer_tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(summarizer_model.device)
  output_ids = summarizer_model.generate(
      **inputs,
      max_new_tokens=45,
      min_new_tokens=15,
      do_sample=False,
    )
  return summarizer_tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [10]:
TRANSLATION_FUNCTION = {
    "name": "translate_arabic_to_english",
    "description": "Translate Arabic text to English",
    "parameters": {
        "type": "object",
        "properties": {
            "text": {
                "type": "string",
                "description": "The text to be translated"
            }
        },
        "required": ["text"],
        "additionalProperties": False
    }
    }

SUMMARY_FUNCTION = {
    "name": "summarize_text",
    "description": "Summarize text",
    "parameters": {
        "type": "object",
        "properties": {
            "text": {
                "type": "string",
                "description": "The text to be summarized"
            }
        },
        "required": ["text"],
        "additionalProperties": False
    }
    }

In [11]:
TOOLS = [
    {"type": "function", "function": TRANSLATION_FUNCTION},
    {"type": "function", "function": SUMMARY_FUNCTION},
]

In [12]:
TOOL_HANDLERS = {
    "translate_arabic_to_english": lambda args: translate_arabic_to_english(args["text"]),
    "summarize_text": lambda args: summarize_text(args["text"]),
}

In [13]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        handler = TOOL_HANDLERS.get(tool_call.function.name)
        if handler is None:
            content = f"Unkown tool: {tool_call.function.name}"
        else:
            arguments = json.loads(tool_call.function.arguments)
            content = handler(arguments)
        responses.append({
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id
            })
    return responses

In [14]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_MESSAGE}] + history + [{"role": "user", "content": message}]
    try:
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)

        while response.choices[0].finish_reason == "tool_calls":
            reply = response.choices[0].message
            messages.append(reply.model_dump(exclude_unset=True))
            messages.extend(handle_tool_calls(reply))
            response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        return response.choices[0].message.content

    except Exception as e:
        print(f"Error calling Gemini {e}")
        return "Something went wrong. Try again Later."

In [15]:
if __name__ == "__main__":
  gr.ChatInterface(fn=chat).launch(inbrowser=True)

d:\Multilingual_news_assistant\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
